# US Regional Sales Analysis

## 1. Notebook Overview

This notebook analyzes the US Regional Sales dataset to evaluate overall business performance and identify patterns in sales, discounts, costs, and profitability.

The analysis builds on the data quality checks and validation performed in the data inspection notebook. It prepares business metrics using documented working assumptions and examines sales performance across different business dimensions.

### Objectives

- Evaluate overall sales performance using key performance indicators (KPIs).
- Compare sales channels and warehouses.
- Analyze product, customer, and sales-team performance.
- Examine profitability and discounting patterns.
- Identify actionable business insights supported by the data.

### Metric Definitions and Assumptions

The dataset does not include an official data dictionary explaining the monetary fields and discount calculation. Therefore, the financial metrics in this notebook use documented working assumptions. These assumptions will be stated alongside the calculations and should not be interpreted as verified accounting definitions.

## 2. Data Loading and Preparation

This section loads the raw sales dataset and prepares the columns required for business analysis. Date and monetary fields are converted to appropriate data types before calculating sales and profitability metrics.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Define the dataset path
data_path = Path('../data/raw/US_Regional_Sales_Data.csv')

# Load the raw dataset
df = pd.read_csv(data_path)

# Create a working copy for analysis
clean_df = df.copy()

print(f"Dataset shape: {clean_df.shape}")
display(clean_df.head())

Dataset shape: (7991, 16)


,OrderNumber,Sales Channel,WarehouseCode,ProcuredDate,OrderDate,ShipDate,DeliveryDate,CurrencyCode,_SalesTeamID,_CustomerID,_StoreID,_ProductID,Order Quantity,Discount Applied,Unit Cost,Unit Price
0,SO - 000101,In-Store,WARE-UHY1004,31-12-2017,31-05-2018,14-06-2018,19-06-2018,USD,6,15,259,12,5,0.075,"1,001.18","1,963.10"
1,SO - 000102,Online,WARE-NMK1003,31-12-2017,31-05-2018,22-06-2018,02-07-2018,USD,14,20,196,27,3,0.075,"3,348.66","3,939.60"
2,SO - 000103,Distributor,WARE-UHY1004,31-12-2017,31-05-2018,21-06-2018,01-07-2018,USD,21,16,213,16,1,0.050,781.22,"1,775.50"
3,SO - 000104,Wholesale,WARE-NMK1003,31-12-2017,31-05-2018,02-06-2018,07-06-2018,USD,28,48,107,23,8,0.075,"1,464.69","2,324.90"
4,SO - 000105,Distributor,WARE-NMK1003,10-04-2018,31-05-2018,16-06-2018,26-06-2018,USD,22,49,111,26,8,0.100,"1,476.14","1,822.40"


## 3. Business Metric Preparation

This section derives key financial metrics, including gross sales, discount amount, net sales, total cost, and profit, using documented working assumptions. The calculated metrics are validated through integrity checks to ensure consistency before proceeding with business performance analysis.

### Metric Definitions and Assumptions

The dataset does not include an official data dictionary explaining the monetary fields and discount calculation. Therefore, the following metric definitions are adopted as working assumptions:

- **Gross Sales:** Order Quantity × Unit Price
- **Discount Amount:** Gross Sales × Discount Applied
- **Net Sales:** Gross Sales − Discount Amount
- **Total Cost:** Order Quantity × Unit Cost
- **Profit:** Net Sales − Total Cost

The `Discount Applied` field is treated as a decimal discount rate based on its observed values. These calculations are provisional working assumptions and should not be interpreted as verified accounting definitions.

In [3]:
# Convert date and monetary columns (strict: raises an error on any bad value)
date_cols = ['ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate']
money_cols = ['Unit Cost', 'Unit Price']

before = clean_df[date_cols + money_cols].dtypes.astype(str)

for col in date_cols:
    clean_df[col] = pd.to_datetime(clean_df[col], format='%d-%m-%Y', errors='raise')

for col in money_cols:
    clean_df[col] = pd.to_numeric(
        clean_df[col].astype(str).str.replace(',', '', regex=False),
        errors='raise'
    )

after = clean_df[date_cols + money_cols].dtypes.astype(str)

report = pd.DataFrame({
    'Type': ['date'] * len(date_cols) + ['money'] * len(money_cols),
    'Dtype before': before,
    'Dtype after': after,
    'Missing after': clean_df[date_cols + money_cols].isna().sum(),
})

print(f"\n{'=' * 65}\nCONVERSION REPORT\n{'=' * 65}")
print(report.to_string())

# Final result
ok_dates = all(clean_df[c].dtype.kind == 'M' for c in date_cols)
ok_money = all(clean_df[c].dtype.kind == 'f' for c in money_cols)

print(f"\n{'=' * 65}\nFINAL RESULT\n{'=' * 65}")
print(f"Columns converted: {len(report)} ({len(date_cols)} date, {len(money_cols)} money)")
print("Verdict:", "All columns converted successfully." if ok_dates and ok_money
      else "Some columns have an unexpected dtype. Check the report above.")


CONVERSION REPORT
               Type Dtype before     Dtype after  Missing after
ProcuredDate   date       object  datetime64[ns]              0
OrderDate      date       object  datetime64[ns]              0
ShipDate       date       object  datetime64[ns]              0
DeliveryDate   date       object  datetime64[ns]              0
Unit Cost     money       object         float64              0
Unit Price    money       object         float64              0

FINAL RESULT
Columns converted: 6 (4 date, 2 money)
Verdict: All columns converted successfully.


In [4]:
# Calculate business metrics using documented working assumptions
clean_df['Gross Sales']     = clean_df['Order Quantity'] * clean_df['Unit Price']
clean_df['Discount Amount'] = clean_df['Gross Sales'] * clean_df['Discount Applied']
clean_df['Net Sales']       = clean_df['Gross Sales'] - clean_df['Discount Amount']
clean_df['Total Cost']      = clean_df['Order Quantity'] * clean_df['Unit Cost']
clean_df['Profit']          = clean_df['Net Sales'] - clean_df['Total Cost']

metric_cols = ['Order Quantity', 'Gross Sales', 'Discount Amount',
               'Net Sales', 'Total Cost', 'Profit']

# 1. Preview
print(f"\n{'=' * 80}\nBUSINESS METRICS (first 10 rows)\n{'=' * 80}")
display(clean_df[metric_cols].head(10).style.format(
    {c: '{:,.2f}' for c in metric_cols if c != 'Order Quantity'}
))

# 2. Integrity checks
money = ['Gross Sales', 'Discount Amount', 'Net Sales', 'Total Cost', 'Profit']
checks = pd.Series({
    'Missing metric values':         int(clean_df[money].isna().any(axis=1).sum()),
    'Net Sales > Gross Sales':       int((clean_df['Net Sales'] > clean_df['Gross Sales']).sum()),
    'Net != Gross - Discount':       int(((clean_df['Gross Sales'] - clean_df['Discount Amount'])
                                          - clean_df['Net Sales']).abs().gt(1e-6).sum()),
    'Loss-making rows (info only)':  int((clean_df['Profit'] < 0).sum()),
}, name='Rows')

print(f"\n{'=' * 80}\nINTEGRITY CHECKS\n{'=' * 80}")
print(checks.to_frame().to_string())

# Final result
errors = int(checks.iloc[:3].sum())
print(f"\n{'=' * 80}\nFINAL RESULT\n{'=' * 80}")
print(f"Metrics created: {len(money)} ({', '.join(money)})")
print(f"Rows calculated: {len(clean_df):,}")
print("Verdict:", "Business metrics calculated correctly." if errors == 0
      else "Calculation issues found. Inspect those rows before continuing.")


BUSINESS METRICS (first 10 rows)


,Order Quantity,Gross Sales,Discount Amount,Net Sales,Total Cost,Profit
0,5,"9,815.50",736.16,"9,079.34","5,005.90","4,073.44"
1,3,"11,818.80",886.41,"10,932.39","10,045.98",886.41
2,1,"1,775.50",88.78,"1,686.72",781.22,905.50
3,8,"18,599.20","1,394.94","17,204.26","11,717.52","5,486.74"
4,8,"14,579.20","1,457.92","13,121.28","11,809.12","1,312.16"
5,5,"5,192.50",259.62,"4,932.88","2,232.80","2,700.07"
6,4,"4,770.40",715.56,"4,054.84","2,146.68","1,908.16"
7,5,"9,078.50",453.93,"8,624.58","7,625.95",998.62
8,4,"15,517.20","4,655.16","10,862.04","8,844.80","2,017.24"
9,8,"15,651.20",782.56,"14,868.64","9,703.76","5,164.88"



INTEGRITY CHECKS
                              Rows
Missing metric values            0
Net Sales > Gross Sales          0
Net != Gross - Discount          0
Loss-making rows (info only)   372

FINAL RESULT
Metrics created: 5 (Gross Sales, Discount Amount, Net Sales, Total Cost, Profit)
Rows calculated: 7,991
Verdict: Business metrics calculated correctly.


In [5]:
# Validate calculated business metrics
money_cols = ['Gross Sales', 'Discount Amount', 'Net Sales', 'Total Cost', 'Profit']
n = len(clean_df)

checks = {
    # (check name, count, counts as an error?)
    'Missing metric values':            (int(clean_df[money_cols].isna().any(axis=1).sum()), True),
    'Net Sales > Gross Sales':          (int((clean_df['Net Sales'] > clean_df['Gross Sales']).sum()), True),
    'Net Sales calculation mismatches': (int(((clean_df['Gross Sales'] - clean_df['Discount Amount']
                                               - clean_df['Net Sales']).abs() >= 1e-6).sum()), True),
    'Negative discount amounts':        (int((clean_df['Discount Amount'] < 0).sum()), True),
    'Negative total costs':             (int((clean_df['Total Cost'] < 0).sum()), True),
    'Loss-making rows':                 (int((clean_df['Profit'] < 0).sum()), False),
}

report = pd.DataFrame(
    [{'Check': k, 'Rows': v, '% of rows': round(v / n * 100, 2),
      'Type': 'Error' if is_err else 'Info'} for k, (v, is_err) in checks.items()]
).set_index('Check')

print(f"\n{'=' * 70}\nBUSINESS METRIC VALIDATION\n{'=' * 70}")
print(report.to_string())

# Final result
errors = int(report.loc[report['Type'] == 'Error', 'Rows'].sum())
loss_rows = checks['Loss-making rows'][0]

print(f"\n{'=' * 70}\nFINAL RESULT\n{'=' * 70}")
print(f"Rows checked:        {n:,}")
print(f"Errors found:        {errors:,}")
print(f"Loss-making rows:    {loss_rows:,} ({loss_rows / n * 100:.2f}%) - business finding, not an error")
print("Verdict:", "All business metrics are valid." if errors == 0
      else "Errors found. Inspect those rows before continuing.")


BUSINESS METRIC VALIDATION
                                  Rows  % of rows   Type
Check                                                   
Missing metric values                0       0.00  Error
Net Sales > Gross Sales              0       0.00  Error
Net Sales calculation mismatches     0       0.00  Error
Negative discount amounts            0       0.00  Error
Negative total costs                 0       0.00  Error
Loss-making rows                   372       4.66   Info

FINAL RESULT
Rows checked:        7,991
Errors found:        0
Loss-making rows:    372 (4.66%) - business finding, not an error
Verdict: All business metrics are valid.


## 4. Overall Sales Performance

This section evaluates the overall performance of the business using key performance indicators (KPIs), including order volume, quantity sold, gross sales, discounts, net sales, total cost, profit, and profitability ratios. These metrics establish a baseline for comparing performance across different business dimensions.

In [6]:
# Summarize overall sales performance
net = clean_df['Net Sales'].sum()
gross = clean_df['Gross Sales'].sum()
profit = clean_df['Profit'].sum()
discounts = clean_df['Discount Amount'].sum()
total_cost = clean_df['Total Cost'].sum()
orders = len(clean_df)

kpis = [
    ('Total Orders',          orders,                           '{:,.0f}'),
    ('Total Quantity Sold',   clean_df['Order Quantity'].sum(), '{:,.0f}'),
    ('Gross Sales',           gross,                            '{:,.2f}'),
    ('Total Discounts',       discounts,                        '{:,.2f}'),
    ('Net Sales',             net,                              '{:,.2f}'),
    ('Total Cost',            total_cost,                       '{:,.2f}'),
    ('Total Profit',          profit,                           '{:,.2f}'),
    ('Profit Margin (%)',     profit / net * 100,               '{:.2f}'),
    ('Avg Discount Rate (%)', discounts / gross * 100,          '{:.2f}'),
    ('Avg Net Sales / Order', net / orders,                     '{:,.2f}'),
    ('Avg Profit / Order',    profit / orders,                  '{:,.2f}'),
]

width = 55
print(f"\n{'=' * width}\nOVERALL SALES PERFORMANCE\n{'=' * width}")
for label, val, fmt in kpis:
    print(f"{label:<28}{fmt.format(val):>{width - 28}}")

# Reconciliation: the totals must tie together
ties = (abs(gross - discounts - net) < 0.01) and (abs(net - total_cost - profit) < 0.01)

print(f"\n{'=' * width}\nFINAL RESULT\n{'=' * width}")
print(f"{orders:,} orders generated {net:,.2f} in net sales "
      f"and {profit:,.2f} in profit ({profit / net * 100:.2f}% margin).")
print("Reconciliation:", "Gross - Discounts = Net, and Net - Cost = Profit." if ties
      else "Totals do not reconcile. Re-check the metric calculations.")


OVERALL SALES PERFORMANCE
Total Orders                                      7,991
Total Quantity Sold                              36,162
Gross Sales                               82,692,726.60
Total Discounts                            9,549,346.86
Net Sales                                 73,143,379.75
Total Cost                                51,818,068.88
Total Profit                              21,325,310.87
Profit Margin (%)                                 29.16
Avg Discount Rate (%)                             11.55
Avg Net Sales / Order                          9,153.22
Avg Profit / Order                             2,668.67

FINAL RESULT
7,991 orders generated 73,143,379.75 in net sales and 21,325,310.87 in profit (29.16% margin).
Reconciliation: Gross - Discounts = Net, and Net - Cost = Profit.


## 5. Sales Performance by Channel

This section compares sales performance across the four sales channels: In-Store, Online, Distributor, and Wholesale. It examines order volume, quantity sold, discounts, net sales, profit, profit margin, and each channel's contribution to overall sales and profitability.

In [7]:
# Sales performance by channel
channel = (
    clean_df.groupby('Sales Channel')
    .agg(
        Orders=('OrderNumber', 'count'),
        Quantity=('Order Quantity', 'sum'),
        Gross_Sales=('Gross Sales', 'sum'),
        Discounts=('Discount Amount', 'sum'),
        Net_Sales=('Net Sales', 'sum'),
        Profit=('Profit', 'sum'),
    )
)

# Channel performance metrics
channel['Margin %']        = channel['Profit'] / channel['Net_Sales'] * 100
channel['Discount Rate %'] = channel['Discounts'] / channel['Gross_Sales'] * 100
channel['Sales Share %']   = channel['Net_Sales'] / channel['Net_Sales'].sum() * 100
channel['Profit Share %']  = channel['Profit'] / channel['Profit'].sum() * 100

channel = channel.sort_values('Net_Sales', ascending=False)

# TOTAL row (built from the sums, so the ratios are correct)
tot = channel[['Orders', 'Quantity', 'Gross_Sales', 'Discounts', 'Net_Sales', 'Profit']].sum()
tot['Margin %']        = tot['Profit'] / tot['Net_Sales'] * 100
tot['Discount Rate %'] = tot['Discounts'] / tot['Gross_Sales'] * 100
tot['Sales Share %']   = 100.0
tot['Profit Share %']  = 100.0

table = pd.concat([channel, tot.to_frame('TOTAL').T]).drop(columns='Gross_Sales')

fmt = {
    'Orders': '{:,.0f}', 'Quantity': '{:,.0f}',
    'Discounts': '{:,.2f}', 'Net_Sales': '{:,.2f}', 'Profit': '{:,.2f}',
    'Margin %': '{:.2f}', 'Discount Rate %': '{:.2f}',
    'Sales Share %': '{:.2f}', 'Profit Share %': '{:.2f}',
}

print(f"\n{'=' * 110}\nSALES PERFORMANCE BY CHANNEL\n{'=' * 110}")
display(
    table.style
         .format(fmt)
         .set_caption('Sales Performance by Channel')
         .apply(lambda r: ['font-weight: bold'] * len(r) if r.name == 'TOTAL' else [''] * len(r), axis=1)
)

# Final result
top_sales  = channel['Net_Sales'].idxmax()
top_margin = channel['Margin %'].idxmax()
low_margin = channel['Margin %'].idxmin()

print(f"\n{'=' * 110}\nFINAL RESULT\n{'=' * 110}")
print(f"Channels analysed:     {len(channel)}")
print(f"Highest net sales:     {top_sales} ({channel.loc[top_sales, 'Sales Share %']:.2f}% of total)")
print(f"Highest profit margin: {top_margin} ({channel.loc[top_margin, 'Margin %']:.2f}%)")
print(f"Lowest profit margin:  {low_margin} ({channel.loc[low_margin, 'Margin %']:.2f}%)")


SALES PERFORMANCE BY CHANNEL


,Orders,Quantity,Discounts,Net_Sales,Profit,Margin %,Discount Rate %,Sales Share %,Profit Share %
In-Store,"3,298","14,878","3,937,209.10","30,102,904.70","8,797,853.23",29.23,11.57,41.16,41.26
Online,"2,425","10,897","2,931,197.24","21,698,558.86","6,166,815.70",28.42,11.90,29.67,28.92
Distributor,"1,375","6,287","1,640,760.15","13,169,147.65","3,887,897.39",29.52,11.08,18.00,18.23
Wholesale,893,"4,100","1,040,180.36","8,172,768.54","2,472,744.54",30.26,11.29,11.17,11.60
TOTAL,"7,991","36,162","9,549,346.85","73,143,379.75","21,325,310.86",29.16,11.55,100.00,100.00



FINAL RESULT
Channels analysed:     4
Highest net sales:     In-Store (41.16% of total)
Highest profit margin: Wholesale (30.26%)
Lowest profit margin:  Online (28.42%)


### Key Findings

1. **In-Store:** Generated the highest net sales, contributing 41.16% of total revenue and 41.26% of total profit.
2. **Wholesale:** Achieved the highest profit margin at 30.26%, indicating stronger profitability per unit of net sales under the adopted metric definitions.
3. **Online:** Recorded the lowest profit margin at 28.42% and the highest discount rate at 11.90%. Further analysis is required to determine whether discounting contributes to its lower margin.
4. **Distributor:** Achieved a profit margin of 29.52%, above the overall business margin of 29.16%.

**Business takeaway:** In-Store is the largest contributor to revenue, while Wholesale leads in profitability margin. Further investigation into Online discounting and costs may help identify opportunities to improve profitability.

## 6. Sales Performance Over Time

This section analyzes monthly sales, profit, order volume, and profit margins to identify trends and variations in business performance.

In [9]:
# Aggregate sales performance by month
monthly = (
    clean_df
    .assign(Month=clean_df['OrderDate'].dt.to_period('M'))
    .groupby('Month')
    .agg(
        Orders=('OrderNumber', 'count'),
        Quantity=('Order Quantity', 'sum'),
        Net_Sales=('Net Sales', 'sum'),
        Profit=('Profit', 'sum'),
    )
)

monthly['Margin %']       = monthly['Profit'] / monthly['Net_Sales'] * 100
monthly['Sales MoM %']    = monthly['Net_Sales'].pct_change() * 100
monthly['Profit MoM %']   = monthly['Profit'].pct_change() * 100
monthly['Sales Share %']  = monthly['Net_Sales'] / monthly['Net_Sales'].sum() * 100

fmt = {
    'Orders': '{:,.0f}', 'Quantity': '{:,.0f}',
    'Net_Sales': '{:,.2f}', 'Profit': '{:,.2f}',
    'Margin %': '{:.2f}', 'Sales MoM %': '{:+.2f}', 'Profit MoM %': '{:+.2f}',
    'Sales Share %': '{:.2f}',
}

print(f"\n{'=' * 100}\nMONTHLY SALES PERFORMANCE\n{'=' * 100}")
display(
    monthly.style
           .format(fmt, na_rep='-')
           .background_gradient(subset=['Net_Sales'], cmap='Blues')
           .set_caption('Monthly Sales Performance')
)

# Final result
best, worst = monthly['Net_Sales'].idxmax(), monthly['Net_Sales'].idxmin()
best_mom, worst_mom = monthly['Sales MoM %'].idxmax(), monthly['Sales MoM %'].idxmin()

print(f"\n{'=' * 100}\nFINAL RESULT\n{'=' * 100}")
print(f"Period covered:       {monthly.index.min()} to {monthly.index.max()} ({len(monthly)} months)")
print(f"Best month:           {best} ({monthly.loc[best, 'Net_Sales']:,.2f} net sales)")
print(f"Worst month:          {worst} ({monthly.loc[worst, 'Net_Sales']:,.2f} net sales)")
print(f"Biggest MoM increase: {best_mom} ({monthly.loc[best_mom, 'Sales MoM %']:+.2f}%)")
print(f"Biggest MoM decrease: {worst_mom} ({monthly.loc[worst_mom, 'Sales MoM %']:+.2f}%)")
print(f"Average monthly net sales: {monthly['Net_Sales'].mean():,.2f}")


MONTHLY SALES PERFORMANCE


,Orders,Quantity,Net_Sales,Profit,Margin %,Sales MoM %,Profit MoM %,Sales Share %
Month,,,,,,,,
2018-05,8,39,"69,636.28","18,271.11",26.24,-,-,0.10
2018-06,252,"1,112","2,180,209.65","629,411.90",28.87,+3030.85,+3344.85,2.98
2018-07,261,"1,211","2,396,750.80","702,408.16",29.31,+9.93,+11.60,3.28
2018-08,279,"1,266","2,563,353.84","730,921.51",28.51,+6.95,+4.06,3.50
2018-09,254,"1,228","2,455,740.78","696,348.45",28.36,-4.20,-4.73,3.36
2018-10,228,"1,072","2,062,013.94","603,902.95",29.29,-16.03,-13.28,2.82
2018-11,257,"1,167","2,661,493.76","774,184.36",29.09,+29.07,+28.20,3.64
2018-12,297,"1,387","2,713,434.00","791,801.84",29.18,+1.95,+2.28,3.71
2019-01,293,"1,297","2,712,820.62","776,917.81",28.64,-0.02,-1.88,3.71



FINAL RESULT
Period covered:       2018-05 to 2020-12 (32 months)
Best month:           2020-01 (2,811,010.63 net sales)
Worst month:          2018-05 (69,636.28 net sales)
Biggest MoM increase: 2018-06 (+3030.85%)
Biggest MoM decrease: 2019-02 (-28.48%)
Average monthly net sales: 2,285,730.62


In [11]:
# Identify strongest and weakest months by net sales
top_n = 3
ranked = monthly_sales['Net_Sales'].sort_values(ascending=False)
avg = monthly_sales['Net_Sales'].mean()

def highlight_table(series):
    return pd.DataFrame({
        'Net Sales': series.map('{:,.2f}'.format),
        'vs Monthly Avg %': ((series / avg - 1) * 100).map('{:+.2f}'.format),
        'Orders': monthly_sales.loc[series.index, 'Orders'].map('{:,.0f}'.format),
    })

best_month, worst_month = ranked.index[0], ranked.index[-1]
spread = ranked.iloc[0] / ranked.iloc[-1]

print(f"\n{'=' * 55}\nTOP {top_n} MONTHS BY NET SALES\n{'=' * 55}")
print(highlight_table(ranked.head(top_n)).to_string())

print(f"\n{'=' * 55}\nBOTTOM {top_n} MONTHS BY NET SALES\n{'=' * 55}")
print(highlight_table(ranked.tail(top_n).sort_values()).to_string())

# Final result
print(f"\n{'=' * 55}\nFINAL RESULT\n{'=' * 55}")
print(f"Highest net sales: {best_month}  ({ranked.iloc[0]:,.2f})")
print(f"Lowest net sales:  {worst_month}  ({ranked.iloc[-1]:,.2f})")
print(f"Best month is {spread:.2f}x the weakest month.")


TOP 3 MONTHS BY NET SALES
            Net Sales vs Monthly Avg % Orders
Month                                        
2020-01  2,811,010.63           +22.98    276
2020-07  2,775,014.71           +21.41    281
2018-12  2,713,434.00           +18.71    297

BOTTOM 3 MONTHS BY NET SALES
            Net Sales vs Monthly Avg % Orders
Month                                        
2018-05     69,636.28           -96.95      8
2019-03  1,936,393.13           -15.28    213
2019-02  1,940,231.39           -15.12    232

FINAL RESULT
Highest net sales: 2020-01  (2,811,010.63)
Lowest net sales:  2018-05  (69,636.28)
Best month is 40.37x the weakest month.


In [14]:
# Inspect monthly order counts and net sales
monthly_sales['Net Sales per Order'] = monthly_sales['Net_Sales'] / monthly_sales['Orders']

# Days in each month that actually have orders
active_days = clean_df.groupby(clean_df['OrderDate'].dt.to_period('M'))['OrderDate'].nunique()

quality = monthly_sales[['Orders', 'Net_Sales', 'Net Sales per Order']].copy()
quality['Days with orders'] = active_days
quality['Days in month']    = quality.index.days_in_month
quality['Coverage %']       = quality['Days with orders'] / quality['Days in month'] * 100
quality['Orders vs median %'] = (quality['Orders'] / quality['Orders'].median() - 1) * 100

# Flag months that may be partial or unusually thin
quality['Flag'] = ((quality['Coverage %'] < 80) | (quality['Orders vs median %'] < -50)).map({True: 'CHECK', False: ''})

fmt = {
    'Orders': '{:,.0f}', 'Net_Sales': '{:,.2f}', 'Net Sales per Order': '{:,.2f}',
    'Days with orders': '{:,.0f}', 'Days in month': '{:,.0f}',
    'Coverage %': '{:.1f}', 'Orders vs median %': '{:+.1f}',
}

print(f"\n{'=' * 100}\nMONTHLY DATA QUALITY CHECK\n{'=' * 100}")
display(
    quality.style
           .format(fmt)
           .apply(lambda r: ['background-color: #ffd6d6; color: #8b0000; font-weight: bold'] * len(r)
              if r['Flag'] else [''] * len(r),
    axis=1
)
           .set_caption('Monthly Data Quality Check')
)

# Final result
flagged = quality.index[quality['Flag'] == 'CHECK'].astype(str).tolist()

print(f"\n{'=' * 100}\nFINAL RESULT\n{'=' * 100}")
print(f"Order dates span:     {clean_df['OrderDate'].min().date()} to {clean_df['OrderDate'].max().date()}")
print(f"Months checked:       {len(quality)}")
print(f"Median orders/month:  {quality['Orders'].median():,.0f}")
print(f"Net sales per order:  {quality['Net Sales per Order'].min():,.2f} to {quality['Net Sales per Order'].max():,.2f}")
print("Verdict:", "No partial or unusually thin months found." if not flagged
      else f"Review these months before drawing trend conclusions: {flagged}")


MONTHLY DATA QUALITY CHECK


,Orders,Net_Sales,Net Sales per Order,Days with orders,Days in month,Coverage %,Orders vs median %,Flag
Month,,,,,,,,
2018-05,8,"69,636.28","8,704.54",1,31,3.2,-96.9,CHECK
2018-06,252,"2,180,209.65","8,651.63",30,30,100.0,-1.4,
2018-07,261,"2,396,750.80","9,182.95",31,31,100.0,+2.2,
2018-08,279,"2,563,353.84","9,187.65",31,31,100.0,+9.2,
2018-09,254,"2,455,740.78","9,668.27",30,30,100.0,-0.6,
2018-10,228,"2,062,013.94","9,043.92",31,31,100.0,-10.8,
2018-11,257,"2,661,493.76","10,356.01",30,30,100.0,+0.6,
2018-12,297,"2,713,434.00","9,136.14",31,31,100.0,+16.2,
2019-01,293,"2,712,820.62","9,258.77",31,31,100.0,+14.7,



FINAL RESULT
Order dates span:     2018-05-31 to 2020-12-30
Months checked:       32
Median orders/month:  256
Net sales per order:  7,951.66 to 10,356.01
Verdict: Review these months before drawing trend conclusions: ['2018-05']


In [16]:
# Inspect the available orders from May 2018
may = clean_df[clean_df['OrderDate'].dt.to_period('M') == '2018-05'].sort_values('OrderDate')

cols = ['OrderNumber', 'OrderDate', 'Sales Channel', 'Order Quantity',
        'Unit Price', 'Discount Applied', 'Net Sales', 'Profit']

print(f"\n{'=' * 90}\nMAY 2018 ORDERS\n{'=' * 90}")
display(
    may[cols].style
        .format({
            'OrderDate': '{:%Y-%m-%d}',
            'Order Quantity': '{:,.0f}',
            'Unit Price': '{:,.2f}',
            'Discount Applied': '{:.1%}',
            'Net Sales': '{:,.2f}',
            'Profit': '{:,.2f}',
        })
        .hide(axis='index')
        .set_caption('May 2018 Orders')
)

# Compare with a typical day in the first full month
june = clean_df[clean_df['OrderDate'].dt.to_period('M') == '2018-06']
june_daily_orders = june.groupby('OrderDate').size().mean()
total_net = clean_df['Net Sales'].sum()

print(f"\n{'=' * 90}\nFINAL RESULT\n{'=' * 90}")
print(f"Orders in May 2018:        {len(may):,}")
print(f"Unique order dates:        {may['OrderDate'].nunique()}")
print(f"Order dates:               {', '.join(may['OrderDate'].dt.strftime('%Y-%m-%d').unique())}")
print(f"Net sales in May 2018:     {may['Net Sales'].sum():,.2f} ({may['Net Sales'].sum() / total_net * 100:.2f}% of total)")
print(f"Avg orders per day (June): {june_daily_orders:,.1f}")
print(f"Avg orders per day (May):  {len(may) / max(may['OrderDate'].nunique(), 1):,.1f}")
print("Verdict:", "May 2018 is a partial month. Exclude it from trend and MoM analysis."
      if may['OrderDate'].nunique() < 20
      else "May 2018 has enough days to keep in the trend analysis.")


MAY 2018 ORDERS


OrderNumber,OrderDate,Sales Channel,Order Quantity,Unit Price,Discount Applied,Net Sales,Profit
SO - 000101,2018-05-31,In-Store,5,"1,963.10",7.5%,"9,079.34","4,073.44"
SO - 000102,2018-05-31,Online,3,"3,939.60",7.5%,"10,932.39",886.41
SO - 000103,2018-05-31,Distributor,1,"1,775.50",5.0%,"1,686.72",905.50
SO - 000104,2018-05-31,Wholesale,8,"2,324.90",7.5%,"17,204.26","5,486.74"
SO - 000105,2018-05-31,Distributor,8,"1,822.40",10.0%,"13,121.28","1,312.16"
SO - 000106,2018-05-31,Online,5,"1,038.50",5.0%,"4,932.88","2,700.07"
SO - 000107,2018-05-31,In-Store,4,"1,192.60",15.0%,"4,054.84","1,908.16"
SO - 000108,2018-05-31,In-Store,5,"1,815.70",5.0%,"8,624.58",998.62



FINAL RESULT
Orders in May 2018:        8
Unique order dates:        1
Order dates:               2018-05-31
Net sales in May 2018:     69,636.28 (0.10% of total)
Avg orders per day (June): 8.4
Avg orders per day (May):  8.0
Verdict: May 2018 is a partial month. Exclude it from trend and MoM analysis.


### Key Findings

- January 2020 recorded the highest net sales at $2,811,010.63.
- May 2018 recorded the lowest net sales at $69,636.28, accounting for 0.10% of total net sales.
- May 2018 contains only 8 orders, all recorded on May 31, indicating partial-month data coverage.
- May 2018 will be excluded from monthly trend and month-over-month comparisons to avoid misleading conclusions.